# ОТЧЕТ ПО РЕЗУЛЬТАТАМ РАЗРАБОТКИ ИНТЕЛЛЕКТУАЛЬНОЙ ПЛАТФОРМЫ
## Модули А-Г: От проектирования до предиктивной аналитики

**Конкурсант:** Анастасия Мужиченкова  
**Компетенция:** Аналитик транспортных данных  
**Дата:** 01.04.2026

---

## 1. Анализ требований и проектирование (Модуль А)

### 1.1 Функциональные и нефункциональные требования
На основе анализа запроса заказчика и конкурсного задания сформулированы следующие требования:

* **Функциональные требования:**
    1. **Детекция и классификация:** Идентификация ТС по 4+ классам (легковые, грузовые, автобусы, мотоциклы) в реальном времени.
    2. **Мониторинг инцидентов:** Автоматический расчет дистанции между ТС и фиксация опасных сближений ($D < 3.5$ м).
    3. **Прогнозирование:** Предсказание плотности потока на горизонт 60 минут.
    4. **Единая версия правды:** Организация единого хранилища (Data Lake) для всех типов данных.

* **Нефункциональные требования:**
    1. **Целостность данных:** Исключение записей с `NULL`-значениями в критических полях (скорость, тип ТС) на уровне витрин.
    2. **Масштабируемость:** Возможность подключения новых видеопотоков без изменения архитектуры БД.
    3. **Актуальность:** Обновление дашбордов в режиме, близком к реальному времени (Near Real-Time).

### 1.2 Обоснование стека технологий
* **СУБД PostgreSQL:** Выбрана как надежное реляционное хранилище с поддержкой сложных аналитических представлений (Views).
* **YOLOv12:** Актуальная модель для высокоскоростной детекции и трекинга объектов.
* **Scikit-learn (Gradient Boosting):** Для реализации прогнозной модели из-за высокой точности на табличных временных рядах.
* **Yandex DataLens:** Для визуализации и обеспечения доступа к "единой версии правды".

---

## 2. Обработка данных и фиксация инцидентов (Модуль Б)

### 2.1 Алгоритм детекции и калибровки
Для перехода от экранных координат к физическим метрам была проведена калибровка участка дороги:
* **Контрольный отрезок:** 15 метров (между виртуальными линиями на кадрах).
* **Расчет:** Вычислен коэффициент `PIXELS_PER_METER`, что позволило перевести расчет скорости и дистанции из пикселей в систему СИ.

### 2.2 Программный триггер сближений
Реализован алгоритм, вычисляющий евклидово расстояние между объектами на каждом кадре. 
> **Оптимизация:** Чтобы избежать избыточности, внедрен "cooldown" механизм: повторная запись одной и той же пары ТС в таблицу `dangerous_incidents` возможна не чаще, чем раз в 2 секунды.

---

## 3. Машинное обучение и прогноз (Модуль В)

### 3.1 Особенности моделирования
Согласно рекомендациям, метрика «интенсивность» была адаптирована под потоковую обработку и